# Tarea C3 — Data Loader Definitivo + Early Stopping
**ECG Rhythm Analyzer | Gabriela — Modelo CNN**  
**Rama:** `feature/model-training`

## Objetivo
Mejorar el pipeline de C2 con tres añadidos importantes:
1. **Data augmentation** en el loader de entrenamiento
2. **Weighted loss** para manejar desbalance de clases
3. **Early stopping** para no desperdiciar tiempo entrenando de más

Todo se prueba con los datos dummy de C2 — en Semana 3 solo cambiamos la ruta.


## PASO 0 — Configuración y GPU

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import os
from PIL import Image

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('⚠️  Sin GPU. Ve a Entorno de ejecución → Cambiar tipo → GPU T4')

## PASO 1 — Recrear los datos dummy de C2
*(Si ya tienes la carpeta `ecg_dataset_dummy` de la sesión anterior, puedes saltarte este paso)*

In [ ]:
# Recrear estructura de carpetas y datos dummy (igual que C2)
IMG_SIZE    = 224
BASE_DIR    = 'ecg_dataset_dummy'
SPLITS      = ['train', 'val', 'test']
CLASES      = ['normal', 'arritmia']
SPLIT_SIZES = {'train': 70, 'val': 20, 'test': 10}

# Crear carpetas
for split in SPLITS:
    for clase in CLASES:
        os.makedirs(os.path.join(BASE_DIR, split, clase), exist_ok=True)

# Generar imágenes si no existen
np.random.seed(42)
total = 0
for split, n_imgs in SPLIT_SIZES.items():
    for clase in CLASES:
        folder = os.path.join(BASE_DIR, split, clase)
        if len(os.listdir(folder)) == 0:  # Solo generar si está vacío
            for i in range(n_imgs):
                pixels = np.random.randint(0, 255, (IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
                Image.fromarray(pixels, 'RGB').save(os.path.join(folder, f'ecg_{split}_{clase}_{i:04d}.png'))
                total += 1

print(f'Datos dummy listos. Imágenes nuevas generadas: {total}')
for split in SPLITS:
    for clase in CLASES:
        n = len(os.listdir(os.path.join(BASE_DIR, split, clase)))
        print(f'  {split}/{clase}: {n} imágenes')

## PASO 2 — Data Loader con Augmentation avanzado

### ¿Por qué augmentation?
El modelo se entrenará con imágenes sintéticas perfectas, pero en la app real recibirá **fotos tomadas con celular** que pueden tener la cámara levemente rotada, diferente iluminación, o un poco de desenfoque. El augmentation simula esas imperfecciones durante el entrenamiento para que el modelo aprenda a manejarlas.

In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
BATCH_SIZE    = 32

# ─── Transform de ENTRENAMIENTO (con augmentation) ───────────
# Cada transformación simula una imperfección de foto real:
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),

    # Rotación leve: simula cámara no perfectamente horizontal
    transforms.RandomRotation(degrees=5),

    # Brillo y contraste: simula diferentes condiciones de iluminación
    transforms.ColorJitter(brightness=0.15, contrast=0.15),

    # Traslación leve: simula ECG no perfectamente centrado en la foto
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),

    # Desenfoque gaussiano leve (opcional): simula foto levemente borrosa
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 0.5)),

    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

# ─── Transform de VALIDACIÓN y TEST (sin augmentation) ───────
# En validación/test NO se augmenta — queremos medir el rendimiento real
val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

# ─── Datasets y DataLoaders ───────────────────────────────────
train_dataset = datasets.ImageFolder(os.path.join(BASE_DIR, 'train'), transform=train_transform)
val_dataset   = datasets.ImageFolder(os.path.join(BASE_DIR, 'val'),   transform=val_transform)
test_dataset  = datasets.ImageFolder(os.path.join(BASE_DIR, 'test'),  transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f'Clases: {train_dataset.classes}  →  0={train_dataset.classes[0]}, 1={train_dataset.classes[1]}')
print(f'Train: {len(train_dataset)} imágenes | Val: {len(val_dataset)} | Test: {len(test_dataset)}')
print(f'Batches por epoch (train): {len(train_loader)}')

In [ ]:
# ─── Visualizar el efecto del augmentation ───────────────────
import matplotlib.pyplot as plt

# Cargar la misma imagen 4 veces para ver cómo varía con augmentation
img_path = os.path.join(BASE_DIR, 'train', 'normal', os.listdir(os.path.join(BASE_DIR, 'train', 'normal'))[0])
img_original = Image.open(img_path)

fig, axes = plt.subplots(1, 5, figsize=(15, 3))
fig.suptitle('Efecto del Data Augmentation — misma imagen, 4 variaciones', fontsize=11)

axes[0].imshow(img_original)
axes[0].set_title('Original', fontsize=9)
axes[0].axis('off')

aug_only = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomRotation(degrees=5),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 0.5)),
])

for i in range(1, 5):
    augmented = aug_only(img_original)
    axes[i].imshow(augmented)
    axes[i].set_title(f'Augmented #{i}', fontsize=9)
    axes[i].axis('off')

plt.tight_layout()
plt.savefig('augmentation_preview.png', dpi=80, bbox_inches='tight')
plt.show()
print('Con imágenes reales de ECG, cada variación simula una foto tomada de forma ligeramente distinta.')

## PASO 3 — Weighted Loss (para desbalance de clases)

### ¿Qué es el desbalance de clases?
En PTB-XL puede haber más registros de arritmia que normales (o al revés). Si el modelo ve 3x más arritmias que normales durante el entrenamiento, aprende a predecir arritmia casi siempre — aunque sea incorrecto. El weighted loss corrige esto penalizando más los errores en la clase minoritaria.

In [ ]:
# ─── Calcular pesos de clases ─────────────────────────────────
# Contar cuántas imágenes hay por clase en el set de entrenamiento
class_counts = {}
for clase in CLASES:
    folder = os.path.join(BASE_DIR, 'train', clase)
    class_counts[clase] = len(os.listdir(folder))

total_train  = sum(class_counts.values())
n_normal     = class_counts['normal']
n_arritmia   = class_counts['arritmia']

print('=== Distribución de clases en Train ===')
print(f'Normal:   {n_normal} imágenes  ({n_normal/total_train*100:.1f}%)')
print(f'Arritmia: {n_arritmia} imágenes  ({n_arritmia/total_train*100:.1f}%)')
print()

# Fórmula: weight = total / (2 × count_clase)
# La clase con menos ejemplos recibe un peso mayor
weight_normal   = total_train / (2 * n_normal)
weight_arritmia = total_train / (2 * n_arritmia)

print('=== Pesos calculados ===')
print(f'Peso normal:   {weight_normal:.4f}')
print(f'Peso arritmia: {weight_arritmia:.4f}')
print()
print('Con datos dummy ambas clases están balanceadas (peso ≈ 1.0).')
print('Con datos reales de PTB-XL los pesos serán distintos.')

# pos_weight en BCEWithLogitsLoss = peso para la clase POSITIVA (arritmia = clase 1)
pos_weight = torch.tensor([weight_arritmia / weight_normal]).to(DEVICE)
criterion  = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

print(f'\npos_weight para BCEWithLogitsLoss: {pos_weight.item():.4f}')
print('✅ Weighted loss configurado.')

## PASO 4 — Early Stopping

### ¿Qué es early stopping?
Sin early stopping, el modelo podría entrenarse 50 epochs aunque dejara de mejorar en la epoch 15. Eso desperdicia tiempo y puede causar overfitting (el modelo memoriza el training set en vez de generalizar). Early stopping detiene el entrenamiento automáticamente si el validation loss no mejora en N epochs consecutivas.

In [ ]:
class EarlyStopping:
    """
    Detiene el entrenamiento si el validation loss no mejora
    después de 'patience' epochs consecutivas.
    También guarda el mejor modelo automáticamente.
    """
    def __init__(self, patience=5, min_delta=0.001, path='mejor_modelo.pt'):
        """
        patience:  cuántas epochs esperar sin mejora antes de parar
        min_delta: mejora mínima que cuenta como 'mejora real'
        path:      dónde guardar el mejor modelo
        """
        self.patience   = patience
        self.min_delta  = min_delta
        self.path       = path
        self.counter    = 0          # cuántas epochs llevamos sin mejorar
        self.best_loss  = None       # mejor val loss visto hasta ahora
        self.stop       = False      # señal de parada

    def __call__(self, val_loss, model):
        """
        Llamar al final de cada epoch con el val_loss actual.
        Retorna True si hay que parar.
        """
        if self.best_loss is None:
            # Primera epoch: guardar como mejor
            self.best_loss = val_loss
            self._guardar_modelo(model)

        elif val_loss < self.best_loss - self.min_delta:
            # Mejoró: resetear contador y guardar modelo
            print(f'    ✅ Val loss mejoró: {self.best_loss:.4f} → {val_loss:.4f}. Guardando modelo.')
            self.best_loss = val_loss
            self.counter   = 0
            self._guardar_modelo(model)

        else:
            # No mejoró: incrementar contador
            self.counter += 1
            print(f'    ⚠️  Sin mejora ({self.counter}/{self.patience}). Mejor loss: {self.best_loss:.4f}')
            if self.counter >= self.patience:
                print(f'    🛑 Early stopping activado después de {self.patience} epochs sin mejora.')
                self.stop = True

        return self.stop

    def _guardar_modelo(self, model):
        """Guarda los pesos del modelo en disco."""
        torch.save(model.state_dict(), self.path)

print('Clase EarlyStopping definida.')
print()
print('Uso:')
print('  early_stopping = EarlyStopping(patience=5)')
print('  if early_stopping(val_loss, model):')
print('      break  ← detiene el loop de epochs')

## PASO 5 — Probar todo junto: 3 epochs con datos dummy

In [ ]:
from torchvision import models

# ─── Modelo ───────────────────────────────────────────────────
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
model.classifier[1] = nn.Linear(1280, 1)
model = model.to(DEVICE)

# ─── Optimizador ──────────────────────────────────────────────
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# ─── Early Stopping ───────────────────────────────────────────
early_stopping = EarlyStopping(patience=5, path='mejor_modelo_c3.pt')

print('Modelo, criterion, optimizer y early stopping listos.')

In [ ]:
# ─── Funciones de train y val (mismas que C2) ─────────────────
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images = images.to(device)
        labels = labels.float().to(device)
        optimizer.zero_grad()
        outputs = model(images).squeeze()
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        preds = (torch.sigmoid(outputs) > 0.5).float()
        correct += (preds == labels).sum().item()
        total   += labels.size(0)
    return total_loss / len(loader), correct / total

def val_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.float().to(device)
            outputs = model(images).squeeze()
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            preds = (torch.sigmoid(outputs) > 0.5).float()
            correct += (preds == labels).sum().item()
            total   += labels.size(0)
    return total_loss / len(loader), correct / total

print('Funciones train_epoch y val_epoch definidas.')

In [ ]:
# ─── Loop de entrenamiento con early stopping ─────────────────
N_EPOCHS = 3  # Solo 3 epochs para la prueba con datos dummy
              # Con datos reales usaremos 30-50 epochs

history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

print('=' * 55)
print('ENTRENAMIENTO C3 (datos dummy, 3 epochs de prueba)')
print('=' * 55)

for epoch in range(1, N_EPOCHS + 1):
    print(f'\n[Epoch {epoch}/{N_EPOCHS}]')

    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, DEVICE)
    val_loss,   val_acc   = val_epoch(model,   val_loader,   criterion, DEVICE)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)

    print(f'  Train → Loss: {train_loss:.4f} | Acc: {train_acc*100:.1f}%')
    print(f'  Val   → Loss: {val_loss:.4f} | Acc: {val_acc*100:.1f}%')

    # Verificar early stopping
    if early_stopping(val_loss, model):
        print('Entrenamiento detenido por early stopping.')
        break

print()
print('=' * 55)
print('✅ Loop de entrenamiento con early stopping funcionando.')
print(f'   Mejor modelo guardado en: mejor_modelo_c3.pt')

In [ ]:
# ─── Graficar curvas de entrenamiento ─────────────────────────
import matplotlib.pyplot as plt

epochs_ran = list(range(1, len(history['train_loss']) + 1))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Curvas de Entrenamiento C3 (datos dummy)', fontsize=12)

# Loss
axes[0].plot(epochs_ran, history['train_loss'], 'b-o', label='Train Loss')
axes[0].plot(epochs_ran, history['val_loss'],   'r-o', label='Val Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Loss por Epoch')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(epochs_ran, [a*100 for a in history['train_acc']], 'b-o', label='Train Acc')
axes[1].plot(epochs_ran, [a*100 for a in history['val_acc']],   'r-o', label='Val Acc')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Accuracy por Epoch')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('curvas_entrenamiento_c3.png', dpi=100, bbox_inches='tight')
plt.show()
print('Con datos reales, esperamos ver train loss bajando suavemente y val loss siguiéndolo.')

## ✅ Checklist C3

In [ ]:
print("""
✅ CHECKLIST C3
================

[✓] Data augmentation implementado en train_transform:
    → RandomRotation(±5°)
    → ColorJitter(brightness=0.15, contrast=0.15)
    → RandomAffine(translate=5%)
    → GaussianBlur leve
    → Val/Test SIN augmentation

[✓] Weighted loss implementado:
    → Peso calculado como total / (2 × count_clase)
    → pos_weight pasado a BCEWithLogitsLoss

[✓] Early stopping implementado:
    → patience=5 (para datos reales)
    → Guarda automáticamente el mejor modelo
    → Integrado en el loop de epochs

[✓] 3 epochs de prueba ejecutadas sin errores
[✓] Curvas de loss y accuracy graficadas

Próximo paso: C4 — Transfer learning en dos fases
""")